# NO2 Nexus Model Walkthrough

This notebook demonstrates a reproducible NO2 downscaling workflow. It uses the included synthetic sample dataset for reviewability, then compares multiple baselines with a spatial holdout to show how the project should be evaluated on real geospatial data.


## Setup

Install the project before running the notebook:

```bash
pip install -e ".[dev]"
```


In [ ]:
from pathlib import Path

import pandas as pd

from no2_nexus.pipeline import (
    NO2Downscaler,
    build_feature_matrix,
    evaluate_models,
    load_csv_dataset,
    save_model_comparison,
)


## Load Sample Data

The sample CSV mimics common NO2 downscaling features: location, coarse NO2, traffic, population density, industrial activity, and a target NO2 value. These values are synthetic and should not be reported as real Sentinel-5P performance.


In [ ]:
data_path = Path('data/sample_no2.csv')
data, target_column = load_csv_dataset(data_path, target='target_NO2')
data.head()


## Train Random Forest Baseline

The Random Forest is the main interpretable nonlinear baseline. It handles mixed-scale tabular features and exposes feature importance.


In [ ]:
features, target = build_feature_matrix(data, target_column)

downscaler = NO2Downscaler(n_estimators=100, random_state=42)
report = downscaler.fit_evaluate(features, target)

pd.DataFrame([report.__dict__])


## Compare Baselines With Spatial Holdout

Spatial validation holds out complete latitude/longitude grid cells. This is more realistic than a random split because nearby geospatial observations can be highly correlated.


In [ ]:
comparison, baseline_predictions = evaluate_models(features, target, split_strategy='spatial')
comparison


## Feature Importance

Feature importance helps explain which inputs drive the Random Forest predictions. On a real dataset, this should be interpreted with domain knowledge and leakage checks.


In [ ]:
downscaler.feature_importances_


## Save Diagnostics

The pipeline exports predictions, baseline comparisons, feature importances, residual plots, and actual-vs-predicted plots.


In [ ]:
output_dir = Path('outputs/notebook_run')
downscaler.save_diagnostics(output_dir)
save_model_comparison(comparison, baseline_predictions, output_dir)
sorted(path.name for path in output_dir.iterdir())


## Limitations

- The included sample data is synthetic. Real claims require exported Sentinel-5P features joined with trusted ground truth.
- Random splits can overstate geospatial performance. Spatial and temporal validation are more credible.
- Sentinel-5P column density and ground-level NO2 concentration are related but physically different targets.
- Production use would need uncertainty estimates, monitoring, and clear data-quality filters.
